## 01 · Preparación de datos de Aduanas

Este notebook descarga los archivos comprimidos desde una carpeta pública de Google Drive, los procesa **ZIP por ZIP**, lee los TXT por bloques (`chunks`), filtra los HS definidos y genera dos archivos procesados:

- `importaciones_hs_filtrado_raw.csv`
- `importaciones_hs_filtrado_estandarizado.csv`

Todo se ejecuta en rutas locales de Colab y al final se descarga automáticamente un ZIP con las salidas.


In [ ]:
# Instalación y librerías
!pip -q install gdown
from pathlib import Path
import pandas as pd
import csv
import time
import zipfile
import shutil
import gdown
from google.colab import files

In [ ]:
# Configuración principal
AUTO_DOWNLOAD_OUTPUTS = True
PUBLIC_GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1N8ZpYM9dsTXuv6FJI2l5HcBnUVsfe0Sq?usp=sharing"
BASE_DIR = Path("/content/proyecto_logistica_cencosud")
TMP_DIR = BASE_DIR / "tmp"
TXT_DIR = BASE_DIR / "txt"
PROCESSED_ADUANAS_DIR = BASE_DIR / "data/processed/aduanas"
EXPORT_DIR = BASE_DIR / "exports"

for carpeta in [TMP_DIR, TXT_DIR, PROCESSED_ADUANAS_DIR, EXPORT_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE_RAW = PROCESSED_ADUANAS_DIR / "importaciones_hs_filtrado_raw.csv"
OUTPUT_FILE_STD = PROCESSED_ADUANAS_DIR / "importaciones_hs_filtrado_estandarizado.csv"
FINAL_ZIP_PATH = EXPORT_DIR / "aduanas_procesadas.zip"

print("Carpeta pública origen:", PUBLIC_GDRIVE_FOLDER_URL)
print("Base local de trabajo:", BASE_DIR)

Carpeta pública origen: https://drive.google.com/drive/folders/1N8ZpYM9dsTXuv6FJI2l5HcBnUVsfe0Sq?usp=sharing
Base local de trabajo: /content/proyecto_logistica_cencosud


In [ ]:
# Encabezados oficiales y HS objetivo
columnas = [
    "NUMENCRIPTADO","TIPO_DOCTO","ADU","FORM","FECVENCI","CODCOMUN","NUM_UNICO_IMPORTADOR",
    "CODPAISCON","DESDIRALM","CODCOMRS","ADUCTROL","NUMPLAZO","INDPARCIAL","NUMHOJINS",
    "TOTINSUM","CODALMA","NUM_RS","FEC_RS","ADUA_RS","NUMHOJANE","NUM_SEC","PA_ORIG","PA_ADQ",
    "VIA_TRAN","TRANSB","PTO_EMB","PTO_DESEM","TPO_CARGA","ALMACEN","FEC_ALMAC","FECRETIRO",
    "NU_REGR","ANO_REG","CODVISBUEN","NUMREGLA","NUMANORES","CODULTVB","PAGO_GRAV","FECTRA",
    "FECACEP","GNOM_CIA_T","CODPAISCIA","NUMRUTCIA","DIGVERCIA","NUM_MANIF","NUM_MANIF1",
    "NUM_MANIF2","FEC_MANIF","NUM_CONOC","FEC_CONOC","NOMEMISOR","NUMRUTEMI","DIGVEREMI",
    "GREG_IMP","REG_IMP","BCO_COM","CODORDIV","FORM_PAGO","NUMDIAS","VALEXFAB","MONEDA",
    "MONGASFOB","CL_COMPRA","TOT_ITEMS","FOB","TOT_HOJAS","COD_FLE","FLETE","TOT_BULTOS",
    "COD_SEG","SEGURO","TOT_PESO","CIF","NUM_AUT","FEC_AUT","GBCOCEN","ID_BULTOS","TPO_BUL1",
    "CANT_BUL1","TPO_BUL2","CANT_BUL2","TPO_BUL3","CANT_BUL3","TPO_BUL4","CANT_BUL4","TPO_BUL5",
    "CANT_BUL5","TPO_BUL6","CANT_BUL6","TPO_BUL7","CANT_BUL7","TPO_BUL8","CANT_BUL8","CTA_OTRO",
    "MON_OTRO","CTA_OTR1","MON_OTR1","CTA_OTR2","MON_OTR2","CTA_OTR3","MON_OTR3","CTA_OTR4",
    "MON_OTR4","CTA_OTR5","MON_OTR5","CTA_OTR6","MON_OTR6","CTA_OTR7","MON_OTR7","MON_178",
    "MON_191","FEC_501","VAL_601","FEC_502","VAL_602","FEC_503","VAL_603","FEC_504","VAL_604",
    "FEC_505","VAL_605","FEC_506","VAL_606","FEC_507","VAL_607","TASA","NCUOTAS","ADU_DI",
    "NUM_DI","FEC_DI","MON_699","MON_199","NUMITEM","DNOMBRE","DMARCA","DVARIEDAD","DOTRO1",
    "DOTRO2","ATR-5","ATR-6","SAJU-ITEM","AJU-ITEM","CANT-MERC","MERMAS","MEDIDA","PRE-UNIT",
    "ARANC-ALA","NUMCOR","NUMACU","CODOBS1","DESOBS1","CODOBS2","DESOBS2","CODOBS3","DESOBS3",
    "CODOBS4","DESOBS4","ARANC-NAC","CIF-ITEM","ADVAL-ALA","ADVAL","VALAD","OTRO1","CTA1",
    "SIGVAL1","VAL1","OTRO2","CTA2","SIGVAL2","VAL2","OTRO3","CTA3","SIGVAL3","VAL3","OTRO4",
    "CTA4","SIGVAL4","VAL4"
]

HS_CODES = {
    '84181019', '84501111', '84501119', '85166030', '85163230', '85163100', '85163290', '85285290',
    '84509000', '84186910', '85285910', '84186990', '85167100', '85167920', '85161035', '84186110',
    '84186940', '85169000', '84182120', '85284200', '85287100', '85164000', '84501132', '85162100',
    '85165000', '85166019', '85167910', '85161050', '85286200', '84186930', '85287300', '84189100',
    '85161039', '84181013', '84186980', '85167930', '84501900', '85167990', '84186970', '85287220',
    '84182130', '84186960', '84501139', '84183000', '84186959', '84501200', '85162900', '84501122',
    '84186120', '85168000', '85287290', '85161034', '84181011', '84189900', '85163300', '84185000',
    '85285920', '84501190', '84501129', '85161049', '84186190', '84182110', '85286900', '84501131',
    '84502000', '85166012', '84182190', '85284900', '85161041', '85285230', '84501112', '85167200',
    '84186951', '84182900', '84181012', '84181090'
}

In [ ]:

# Funciones auxiliares mínimas de limpieza
def normalizar_hs_serie(serie):
    return (
        serie.fillna("")
        .astype(str)
        .str.replace(r"\D", "", regex=True)
        .str[:8]
        .str.strip()
    )

def convertir_numerico(serie):
    return pd.to_numeric(
        serie.fillna("")
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip(),
        errors="coerce"
    )

def convertir_fecha(serie):
    s = (
        serie.fillna("")
        .astype(str)
        .str.strip()
        .str.replace("/", "-", regex=False)
    )
    fecha = pd.to_datetime(s, format="%d-%m-%Y", errors="coerce")
    mask = fecha.isna()
    if mask.any():
        fecha.loc[mask] = pd.to_datetime(s[mask], format="%d-%m-%Y %H:%M:%S", errors="coerce")
    mask = fecha.isna()
    if mask.any():
        fecha.loc[mask] = pd.to_datetime(s[mask], format="%Y-%m-%d", errors="coerce")
    mask = fecha.isna()
    if mask.any():
        fecha.loc[mask] = pd.to_datetime(s[mask], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    mask = fecha.isna()
    if mask.any():
        fecha.loc[mask] = pd.to_datetime(s[mask], format="%d%m%Y", errors="coerce")
    return fecha

## Procesamiento incremental

La lógica del notebook se mantiene simple:

1. descarga la carpeta pública con los ZIP,
2. procesa **ZIP por ZIP**,
3. extrae TXT temporalmente,
4. lee por bloques,
5. filtra temprano por HS,
6. guarda incrementalmente las salidas,
7. limpia temporales.


In [ ]:

# Limpieza previa y parámetros de lectura
for archivo_salida in [OUTPUT_FILE_RAW, OUTPUT_FILE_STD, FINAL_ZIP_PATH]:
    if archivo_salida.exists():
        archivo_salida.unlink()

if TMP_DIR.exists():
    shutil.rmtree(TMP_DIR)
TMP_DIR.mkdir(parents=True, exist_ok=True)

if TXT_DIR.exists():
    shutil.rmtree(TXT_DIR)
TXT_DIR.mkdir(parents=True, exist_ok=True)

chunk_size = 100_000
separador_entrada = ";"
encoding_entrada = "latin1"

inicio = time.time()
total_filtrado = 0
archivos_ok = 0
archivos_error = []
primer_escritura_raw = True
primer_escritura_std = True

print("=" * 70)
print("INICIO DEL PROCESO DE PREPARACIÓN DE DATOS")
print("=" * 70)
print("Descargando carpeta pública con ZIP...")


INICIO DEL PROCESO DE PREPARACIÓN DE DATOS
Descargando carpeta pública con ZIP...


In [ ]:

# Descarga de la carpeta pública y detección de ZIP
descargados = gdown.download_folder(
    url=PUBLIC_GDRIVE_FOLDER_URL,
    output=str(TMP_DIR),
    quiet=False,
    use_cookies=False,
    remaining_ok=True
)

zip_files = sorted(TMP_DIR.rglob("*.zip"))

if not zip_files:
    raise FileNotFoundError("No se encontraron archivos ZIP en la carpeta pública compartida.")

print(f"ZIP encontrados: {len(zip_files)}")
for z in zip_files:
    print(" -", z.name)


Retrieving folder contents


Processing file 1kfSAHVJJEHJm8P2s2_US5K5Kh7RW6k5r Txt_Originales-20260406T214742Z-3-001.zip
Processing file 135RMoFi6ydAXINYTG1jpcPO0am-0hxp0 Txt_Originales-20260406T214742Z-3-002.zip
Processing file 1_5ebhdDhPmeLj4Oq7h1J8EQ7NWSJe-gN Txt_Originales-20260406T214742Z-3-003.zip
Processing file 1u1zhrrsaytJpwZ-YjJskYtGPkzB7fjNl Txt_Originales-20260406T214742Z-3-004.zip
Processing file 1IPRSAU_4zIQR2EaO2g_ELzgn0bXkM1Tp Txt_Originales-20260406T214742Z-3-005.zip
Processing file 1v5sywD4zve-ZaLiYrYJaQKn-1ofrr46s Txt_Originales-20260406T214742Z-3-006.zip
Processing file 1bU_KpkJaG22NjsozIlEafFX0VdXxaJki Txt_Originales-20260406T214742Z-3-007.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1kfSAHVJJEHJm8P2s2_US5K5Kh7RW6k5r
From (redirected): https://drive.google.com/uc?id=1kfSAHVJJEHJm8P2s2_US5K5Kh7RW6k5r&confirm=t&uuid=34e68585-2520-4131-9ac5-4672bbd6552f
To: /content/proyecto_logistica_cencosud/tmp/Txt_Originales-20260406T214742Z-3-001.zip
100%|██████████| 212M/212M [00:00<00:00, 257MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=135RMoFi6ydAXINYTG1jpcPO0am-0hxp0
From (redirected): https://drive.google.com/uc?id=135RMoFi6ydAXINYTG1jpcPO0am-0hxp0&confirm=t&uuid=815ced92-831e-4f96-af7a-ae5d022be392
To: /content/proyecto_logistica_cencosud/tmp/Txt_Originales-20260406T214742Z-3-002.zip
100%|██████████| 213M/213M [00:00<00:00, 251MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1_5ebhdDhPmeLj4Oq7h1J8EQ7NWSJe-gN
From (redirected): https://drive.google.com/uc?id=1_5ebhdD

ZIP encontrados: 7
 - Txt_Originales-20260406T214742Z-3-001.zip
 - Txt_Originales-20260406T214742Z-3-002.zip
 - Txt_Originales-20260406T214742Z-3-003.zip
 - Txt_Originales-20260406T214742Z-3-004.zip
 - Txt_Originales-20260406T214742Z-3-005.zip
 - Txt_Originales-20260406T214742Z-3-006.zip
 - Txt_Originales-20260406T214742Z-3-007.zip


In [ ]:
# Proceso principal ZIP por ZIP
for zip_file in zip_files:
    print("\n" + "=" * 70)
    print(f"Procesando ZIP: {zip_file.name}")
    print("=" * 70)

    if TXT_DIR.exists():
        shutil.rmtree(TXT_DIR)
    TXT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_file, "r") as zf:
        zf.extractall(TXT_DIR)

    archivos_txt = sorted(TXT_DIR.rglob("*.txt"))
    if not archivos_txt:
        print("No se encontraron TXT en este ZIP.")
        continue

    print(f"TXT encontrados en este ZIP: {len(archivos_txt)}")

    for archivo in archivos_txt:
        print(f"\nProcesando TXT: {archivo.name}")

        try:
            lector = pd.read_csv(
                archivo,
                sep=separador_entrada,
                encoding=encoding_entrada,
                header=None,
                dtype=str,
                chunksize=chunk_size,
                engine="python",
                quoting=csv.QUOTE_NONE,
                on_bad_lines="skip"
            )

            for i, chunk in enumerate(lector, start=1):
                if len(chunk) > 0:
                    fila0 = [str(x).strip() if pd.notna(x) else "" for x in chunk.iloc[0].tolist()]
                    if fila0[:len(columnas)] == columnas:
                        chunk = chunk.iloc[1:].copy()

                if chunk.empty:
                    continue

                if chunk.shape[1] < len(columnas):
                    chunk = chunk.reindex(columns=range(len(columnas)))
                elif chunk.shape[1] > len(columnas):
                    chunk = chunk.iloc[:, :len(columnas)].copy()

                chunk.columns = columnas
                chunk["archivo_origen"] = archivo.name

                chunk["HS_CODE"] = normalizar_hs_serie(chunk["ARANC-NAC"])
                filtrado = chunk[chunk["HS_CODE"].isin(HS_CODES)].copy()

                if filtrado.empty:
                    print(f"  Bloque {i}: 0 filas filtradas | Acumulado: {total_filtrado:,}")
                    continue

                filtrado.to_csv(
                    OUTPUT_FILE_RAW,
                    sep=";",
                    mode="w" if primer_escritura_raw else "a",
                    index=False,
                    header=primer_escritura_raw,
                    encoding="utf-8-sig"
                )
                primer_escritura_raw = False

                df_std = pd.DataFrame()
                df_std["archivo_origen"] = filtrado["archivo_origen"]
                df_std["documento_id"] = filtrado["NUMENCRIPTADO"]
                df_std["num_item"] = filtrado["NUMITEM"]
                df_std["fecha"] = convertir_fecha(filtrado["FECTRA"])
                df_std["codigo_arancel"] = filtrado["ARANC-NAC"]
                df_std["hs_code"] = filtrado["HS_CODE"]
                df_std["hs4"] = filtrado["HS_CODE"].astype(str).str[:4]
                df_std["descripcion_producto"] = filtrado["DNOMBRE"]
                df_std["marca"] = filtrado["DMARCA"]
                df_std["unidad_medida"] = filtrado["MEDIDA"]
                df_std["cantidad"] = convertir_numerico(filtrado["CANT-MERC"])
                df_std["peso"] = convertir_numerico(filtrado["TOT_PESO"])
                df_std["valor_cif"] = convertir_numerico(filtrado["CIF-ITEM"])

                desc = df_std["descripcion_producto"].fillna("").astype(str).str.upper().str.strip().str[:40]
                marca = df_std["marca"].fillna("").astype(str).str.upper().str.strip()
                hs = df_std["codigo_arancel"].fillna("").astype(str).str.strip()
                df_std["sku_referencia"] = hs + "_" + marca + "_" + desc

                df_std.to_csv(
                    OUTPUT_FILE_STD,
                    sep=";",
                    mode="w" if primer_escritura_std else "a",
                    index=False,
                    header=primer_escritura_std,
                    encoding="utf-8-sig"
                )
                primer_escritura_std = False

                total_filtrado += len(filtrado)
                print(f"  Bloque {i}: {len(filtrado):,} filas filtradas | Acumulado: {total_filtrado:,}")

            archivos_ok += 1

        except Exception as e:
            print(f"  Error en {archivo.name}: {e}")
            archivos_error.append((archivo.name, str(e)))

    shutil.rmtree(TXT_DIR)
    TXT_DIR.mkdir(parents=True, exist_ok=True)



Procesando ZIP: Txt_Originales-20260406T214742Z-3-001.zip
TXT encontrados en este ZIP: 6

Procesando TXT: Importaciones enero 2025.txt
  Bloque 1: 1,472 filas filtradas | Acumulado: 1,472
  Bloque 2: 1,395 filas filtradas | Acumulado: 2,867
  Bloque 3: 729 filas filtradas | Acumulado: 3,596
  Bloque 4: 946 filas filtradas | Acumulado: 4,542

Procesando TXT: Importaciones febrero 2025.txt
  Bloque 1: 1,270 filas filtradas | Acumulado: 5,812
  Bloque 2: 1,419 filas filtradas | Acumulado: 7,231
  Bloque 3: 535 filas filtradas | Acumulado: 7,766
  Bloque 4: 681 filas filtradas | Acumulado: 8,447

Procesando TXT: Importaciones julio 2025.txt
  Bloque 1: 1,256 filas filtradas | Acumulado: 9,703
  Bloque 2: 1,789 filas filtradas | Acumulado: 11,492
  Bloque 3: 591 filas filtradas | Acumulado: 12,083
  Bloque 4: 701 filas filtradas | Acumulado: 12,784
  Bloque 5: 553 filas filtradas | Acumulado: 13,337

Procesando TXT: Importaciones junio 2023.txt
  Bloque 1: 1,282 filas filtradas | Acumulado

In [ ]:
# Resumen del proceso
fin = time.time()
print("\n" + "=" * 70)
print("PROCESO TERMINADO")
print("=" * 70)
print(f"Archivos TXT procesados correctamente: {archivos_ok}")
print(f"Total registros filtrados: {total_filtrado:,}")
print(f"Archivo raw filtrado: {OUTPUT_FILE_RAW}")
print(f"Archivo estandarizado: {OUTPUT_FILE_STD}")
print(f"Tiempo total: {round(fin - inicio, 2)} segundos")

if archivos_error:
    print("\nArchivos con error:")
    for nombre, error in archivos_error:
        print(f"- {nombre}: {error}")


PROCESO TERMINADO
Archivos TXT procesados correctamente: 36
Total registros filtrados: 147,448
Archivo raw filtrado: /content/proyecto_logistica_cencosud/data/processed/aduanas/importaciones_hs_filtrado_raw.csv
Archivo estandarizado: /content/proyecto_logistica_cencosud/data/processed/aduanas/importaciones_hs_filtrado_estandarizado.csv
Tiempo total: 699.54 segundos


## Validación rápida de salidas

In [ ]:

for archivo in [OUTPUT_FILE_RAW, OUTPUT_FILE_STD]:
    if not archivo.exists():
        print(f"No existe: {archivo}")
        continue

    print("=" * 70)
    print(f"Archivo: {archivo.name}")
    muestra = pd.read_csv(archivo, sep=";", nrows=5, encoding="utf-8-sig")
    print(f"Columnas: {len(muestra.columns)}")
    print(f"Primeras columnas: {muestra.columns.tolist()[:10]}")
    display(muestra.head())


Archivo: importaciones_hs_filtrado_raw.csv
Columnas: 180
Primeras columnas: ['NUMENCRIPTADO', 'TIPO_DOCTO', 'ADU', 'FORM', 'FECVENCI', 'CODCOMUN', 'NUM_UNICO_IMPORTADOR', 'CODPAISCON', 'DESDIRALM', 'CODCOMRS']


,NUMENCRIPTADO,TIPO_DOCTO,ADU,FORM,FECVENCI,CODCOMUN,NUM_UNICO_IMPORTADOR,CODPAISCON,DESDIRALM,CODCOMRS,...,OTRO3,CTA3,SIGVAL3,VAL3,OTRO4,CTA4,SIGVAL4,VAL4,archivo_origen,HS_CODE
0,22075656,151,39,15,5022025,13101,11841,336,NaN,NaN,...,0,NaN,NaN,0,0,NaN,NaN,0,Importaciones enero 2025.txt,85167200
1,22100474,151,39,15,7022025,13101,11841,336,NaN,NaN,...,0,NaN,NaN,0,0,NaN,NaN,0,Importaciones enero 2025.txt,85166019
2,22040085,151,34,15,15022025,13101,8905,336,NaN,NaN,...,0,NaN,NaN,0,0,NaN,NaN,0,Importaciones enero 2025.txt,85167990
3,22040085,151,34,15,15022025,13101,8905,336,NaN,NaN,...,0,NaN,NaN,0,0,NaN,NaN,0,Importaciones enero 2025.txt,85167200
4,22103441,101,48,15,31012025,13123,10144,225,NaN,13124.0,...,0,NaN,NaN,0,0,NaN,NaN,0,Importaciones enero 2025.txt,84189900


Archivo: importaciones_hs_filtrado_estandarizado.csv
Columnas: 14
Primeras columnas: ['archivo_origen', 'documento_id', 'num_item', 'fecha', 'codigo_arancel', 'hs_code', 'hs4', 'descripcion_producto', 'marca', 'unidad_medida']


,archivo_origen,documento_id,num_item,fecha,codigo_arancel,hs_code,hs4,descripcion_producto,marca,unidad_medida,cantidad,peso,valor_cif,sku_referencia
0,Importaciones enero 2025.txt,22075656,18,2025-01-21,85167200,85167200,8516,SIN-CODIGO ~TOSTADOR,WELFN-F,10,48.0000,9390.0,314.18,85167200_WELFN-F_SIN-CODIGO ~TOSTADOR
1,Importaciones enero 2025.txt,22100474,13,2025-01-23,85166019,85166019,8516,SIN-CODIGO ~FREIDORA DE AIRE,WELFN-F,10,50.0000,8580.0,525.22,85166019_WELFN-F_SIN-CODIGO ~FREIDORA DE ...
2,Importaciones enero 2025.txt,22040085,21,2025-01-31,85167990,85167990,8516,SIN-CODIGO ~FREIDORA,YIWU RUN LAI-F,10,56.0000,6700.0,753.63,85167990_YIWU RUN LAI-F_SIN-CODIGO ~FREIDORA
3,Importaciones enero 2025.txt,22040085,25,2025-01-31,85167200,85167200,8516,SIN-CODIGO ~TOSTADORA,YIWU RUN LAI-F,10,120.0000,6700.0,583.90,85167200_YIWU RUN LAI-F_SIN-CODIGO ~TOSTA...
4,Importaciones enero 2025.txt,22103441,1,2025-01-16,84189900,84189900,8418,SIN-CODIGO ~CALENTADOR,MIDWEST-F,6,1.2077,3.9,1482.26,84189900_MIDWEST-F_SIN-CODIGO ~CALENTADOR


## Empaquetado final y descarga automática

In [ ]:

if FINAL_ZIP_PATH.exists():
    FINAL_ZIP_PATH.unlink()

with zipfile.ZipFile(FINAL_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    if OUTPUT_FILE_RAW.exists():
        zf.write(OUTPUT_FILE_RAW, arcname=OUTPUT_FILE_RAW.name)
    if OUTPUT_FILE_STD.exists():
        zf.write(OUTPUT_FILE_STD, arcname=OUTPUT_FILE_STD.name)

print(f"ZIP final generado: {FINAL_ZIP_PATH}")

if AUTO_DOWNLOAD_OUTPUTS and FINAL_ZIP_PATH.exists():
    files.download(str(FINAL_ZIP_PATH))


ZIP final generado: /content/proyecto_logistica_cencosud/exports/aduanas_procesadas.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>